# rag-redteam : attaques contre l'agent sur GPU

**Avant de commencer** : menu *Exécution* → *Modifier le type d'exécution* → **GPU T4**.

Ensuite, lance les cellules une par une, de haut en bas. La cellule 4 te demandera d'envoyer `rag-redteam.zip`.

In [ ]:
# 1. Vérifier le GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Installer et démarrer Ollama, télécharger le modèle de l'agent (~5 min)
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama pull qwen2.5:7b
!ollama list

In [ ]:
# 3. Récupérer merimee-rag, construire le corpus et les index (~5 min)
%cd /content
!git clone -q https://github.com/ezechiel-sawadogo-nlp/merimee-rag.git
%cd /content/merimee-rag
!pip install -q -e .
!merimee-rag prepare
!merimee-rag index

# Correction D3 (sans effet si elle est déjà dans le dépôt)
import re
p = "/content/merimee-rag/src/merimee_rag/agent.py"
s = open(p, encoding="utf-8").read()
s2 = re.sub(r"\n[ \t]*elif isinstance\(x, str\):\r?\n[ \t]*found\.update\(REF_RE\.findall\(x\)\)\r?", "", s)
open(p, "w", encoding="utf-8").write(s2)
print("Correction D3 :", "appliquée" if s2 != s else "déjà présente")

In [ ]:
# 4. Envoyer rag-redteam.zip (fait sur ton PC, voir les instructions)
from google.colab import files
%cd /content
up = files.upload()
!rm -rf /content/rag-redteam && mkdir -p /content/rag-redteam
!unzip -q -o rag-redteam.zip -d /content/rag-redteam
!ls /content/rag-redteam/src/rag_redteam

In [ ]:
# 5. Lancer les 4 attaques contre l'agent (~15 min)
%cd /content/rag-redteam
!python src/rag_redteam/run_agent.py --n 10

In [ ]:
# 6. Récupérer les résultats
!cd /content/rag-redteam && zip -q -r /content/resultats_agent.zip results/agent__*
files.download("/content/resultats_agent.zip")

**Si Ollama ne répond plus** (erreur `ConnectionError`) : relance la cellule 2, puis la cellule 5. Tout ce qui a déjà été calculé sort du cache.